# Identify Customer Segments

**Author:** Sam Sepassi

Udacity / Arvato project using unsupervised learning to compare the general German population against customers of a mail-order company.

**Submission note:** the CSV data is proprietary and must not be committed to GitHub. This notebook assumes the CSV files are present locally in this folder or in Udacity's workspace.

## Rubric Map

This notebook is organized to satisfy the Udacity rubric:

1. Preprocessing: missing-value encoding, column/row missingness, mixed/categorical features, reusable cleaning function.
2. Feature transformation: imputation, scaling, PCA, variance justification, component interpretation.
3. Clustering: KMeans model selection, population vs customer cluster comparison, segment interpretation.

In [ ]:
import ast
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)

DATA_DIR = Path(".")
AZDIAS_PATH = DATA_DIR / "Udacity_AZDIAS_Subset.csv"
CUSTOMERS_PATH = DATA_DIR / "Udacity_CUSTOMERS_Subset.csv"
FEATURE_SUMMARY_PATH = DATA_DIR / "AZDIAS_Feature_Summary.csv"

In [ ]:
# Load data. If this cell fails, place the proprietary CSV files in this folder or run in the Udacity workspace.
azdias = pd.read_csv(AZDIAS_PATH, sep=';')
customers = pd.read_csv(CUSTOMERS_PATH, sep=';')
feat_info = pd.read_csv(FEATURE_SUMMARY_PATH, sep=';')

print("AZDIAS:", azdias.shape)
print("Customers:", customers.shape)
print("Feature summary:", feat_info.shape)
feat_info.head()

## Step 1: Preprocessing

### 1.1 Convert missing / unknown values to NaN

The feature summary file provides encoded missing values per attribute. The helper below parses those codes and replaces matching dataset values with `NaN`.

In [ ]:
def parse_missing_codes(value):
    """Parse the `missing_or_unknown` field from AZDIAS_Feature_Summary.csv."""
    if pd.isna(value):
        return []
    text = str(value).strip()
    if text in ("", "[]"):
        return []
    text = text.strip("[]")
    if not text:
        return []
    codes = []
    for item in text.split(','):
        item = item.strip()
        if item == "":
            continue
        try:
            codes.append(int(item))
        except ValueError:
            try:
                codes.append(float(item))
            except ValueError:
                codes.append(item)
    return codes

missing_map = {
    row.attribute: parse_missing_codes(row.missing_or_unknown)
    for row in feat_info.itertuples(index=False)
}

def convert_missing_to_nan(df, missing_map):
    df = df.copy()
    for col, codes in missing_map.items():
        if col in df.columns and codes:
            df[col] = df[col].replace(codes, np.nan)
    return df

azdias_nan = convert_missing_to_nan(azdias, missing_map)
customers_nan = convert_missing_to_nan(customers, missing_map)

azdias_nan.isna().sum().sort_values(ascending=False).head(10)

### 1.2 Assess missing values by column

Use the distribution below to choose a threshold for dropping columns with unusually high missingness. A common passing approach is to drop clear outliers rather than blindly choosing a threshold.

In [ ]:
col_missing = azdias_nan.isna().mean().sort_values(ascending=False)

plt.figure(figsize=(12, 4))
sns.histplot(col_missing, bins=30)
plt.xlabel("Proportion missing")
plt.ylabel("Number of columns")
plt.title("Missingness by Column")
plt.show()

col_missing.head(20)

In [ ]:
# Adjust after inspecting the histogram. 0.20 is a common starting threshold for this project.
COL_MISSING_THRESHOLD = 0.20
high_missing_cols = col_missing[col_missing > COL_MISSING_THRESHOLD].index.tolist()
print(f"Dropping {len(high_missing_cols)} columns above {COL_MISSING_THRESHOLD:.0%} missingness:")
high_missing_cols

**Discussion:** Explain why the threshold above is appropriate based on the plot and table. Mention whether the removed columns are clear outliers in missingness.

### 1.3 Assess missing values by row

Split rows into lower-missingness and higher-missingness groups, then compare selected feature distributions to determine whether high-missing rows look qualitatively different.

In [ ]:
azdias_cols_clean = azdias_nan.drop(columns=high_missing_cols)
row_missing = azdias_cols_clean.isna().sum(axis=1)

plt.figure(figsize=(12, 4))
sns.histplot(row_missing, bins=50)
plt.xlabel("Missing values per row")
plt.ylabel("Number of rows")
plt.title("Missingness by Row")
plt.show()

row_missing.describe()

In [ ]:
# Adjust after inspecting histogram. Many Udacity solutions use a cutoff around 20-30 missing values.
ROW_MISSING_THRESHOLD = 20
low_missing_rows = azdias_cols_clean[row_missing <= ROW_MISSING_THRESHOLD]
high_missing_rows = azdias_cols_clean[row_missing > ROW_MISSING_THRESHOLD]

print("Low-missing rows:", low_missing_rows.shape)
print("High-missing rows:", high_missing_rows.shape)

compare_cols = [c for c in low_missing_rows.columns if low_missing_rows[c].nunique(dropna=True) <= 10][:6]
for col in compare_cols:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
    low_missing_rows[col].value_counts(normalize=True, dropna=False).sort_index().plot(kind='bar', ax=axes[0], title=f"Low missing: {col}")
    high_missing_rows[col].value_counts(normalize=True, dropna=False).sort_index().plot(kind='bar', ax=axes[1], title=f"High missing: {col}")
    plt.tight_layout()
    plt.show()

**Discussion:** Describe whether high-missing rows appear qualitatively different. For final modeling, use the lower-missingness subset from the general population.

### 1.4 Process categorical and mixed-type features

Use `feat_info.type` to identify categorical and mixed features. Binary categoricals can usually be kept/re-encoded. Multi-level categoricals should be one-hot encoded or dropped. Mixed features should be engineered into numeric components or dropped.

In [ ]:
feat_info_clean = feat_info[~feat_info.attribute.isin(high_missing_cols)].copy()
cat_cols = feat_info_clean.loc[feat_info_clean.type == 'categorical', 'attribute'].tolist()
mixed_cols = feat_info_clean.loc[feat_info_clean.type == 'mixed', 'attribute'].tolist()

cat_cols = [c for c in cat_cols if c in low_missing_rows.columns]
mixed_cols = [c for c in mixed_cols if c in low_missing_rows.columns]

print("Categorical columns:", cat_cols)
print("Mixed columns:", mixed_cols)

for col in cat_cols:
    print(col, low_missing_rows[col].dropna().unique()[:20], "nunique=", low_missing_rows[col].nunique(dropna=True))

In [ ]:
def engineer_mixed_features(df):
    """Engineer known mixed features from the Udacity Arvato dataset."""
    df = df.copy()

    # PRAEGENDE_JUGENDJAHRE combines decade and movement.
    if 'PRAEGENDE_JUGENDJAHRE' in df.columns:
        decade_map = {
            1: 40, 2: 40,
            3: 50, 4: 50,
            5: 60, 6: 60, 7: 60,
            8: 70, 9: 70,
            10: 80, 11: 80, 12: 80, 13: 80,
            14: 90, 15: 90,
        }
        movement_map = {
            1: 0, 3: 0, 5: 0, 8: 0, 10: 0, 12: 0, 14: 0,  # mainstream
            2: 1, 4: 1, 6: 1, 7: 1, 9: 1, 11: 1, 13: 1, 15: 1,  # avantgarde
        }
        df['PRAEGENDE_JUGENDJAHRE_DECADE'] = df['PRAEGENDE_JUGENDJAHRE'].map(decade_map)
        df['PRAEGENDE_JUGENDJAHRE_MOVEMENT'] = df['PRAEGENDE_JUGENDJAHRE'].map(movement_map)
        df = df.drop(columns=['PRAEGENDE_JUGENDJAHRE'])

    # CAMEO_INTL_2015 combines wealth and life stage; values can be strings like '51'.
    if 'CAMEO_INTL_2015' in df.columns:
        cameo = df['CAMEO_INTL_2015'].astype('string')
        df['CAMEO_INTL_2015_WEALTH'] = pd.to_numeric(cameo.str[0], errors='coerce')
        df['CAMEO_INTL_2015_LIFE_STAGE'] = pd.to_numeric(cameo.str[1], errors='coerce')
        df = df.drop(columns=['CAMEO_INTL_2015'])

    # Drop remaining mixed columns unless separately engineered.
    remaining_mixed = [c for c in mixed_cols if c in df.columns]
    df = df.drop(columns=remaining_mixed, errors='ignore')
    return df


def process_categorical_features(df):
    df = df.copy()

    # Recode OST_WEST_KZ from W/O to numeric if present.
    if 'OST_WEST_KZ' in df.columns:
        df['OST_WEST_KZ'] = df['OST_WEST_KZ'].map({'W': 0, 'O': 1})

    current_cat_cols = [c for c in cat_cols if c in df.columns]
    binary_cols = [c for c in current_cat_cols if df[c].nunique(dropna=True) <= 2]
    multi_cols = [c for c in current_cat_cols if df[c].nunique(dropna=True) > 2]

    # Keep binary columns as-is after numeric conversion where possible; one-hot encode multi-level categoricals.
    df = pd.get_dummies(df, columns=multi_cols, dummy_na=False, drop_first=False)

    for col in binary_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    return df

### 1.5 Create reusable cleaning function

The same cleaning choices from the general population must be applied to customers.

In [ ]:
def clean_data(df):
    """Clean Arvato demographic data using decisions fit on the AZDIAS population data."""
    df = convert_missing_to_nan(df, missing_map)
    df = df.drop(columns=high_missing_cols, errors='ignore')

    row_missing = df.isna().sum(axis=1)
    df = df[row_missing <= ROW_MISSING_THRESHOLD].copy()

    df = engineer_mixed_features(df)
    df = process_categorical_features(df)

    # Ensure every remaining column is numeric and analysis-ready.
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    return df

azdias_clean = clean_data(azdias)
customers_clean = clean_data(customers)

# Align columns so customers has exactly the same feature matrix as general population.
customers_clean = customers_clean.reindex(columns=azdias_clean.columns, fill_value=0)

print("Clean AZDIAS:", azdias_clean.shape)
print("Clean customers:", customers_clean.shape)
azdias_clean.head()

## Step 2: Feature Transformation

Algorithms like PCA and KMeans use distance/variance. Without scaling, features with larger numeric ranges dominate the model even when they are not more important.

In [ ]:
imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

azdias_imputed = imputer.fit_transform(azdias_clean)
azdias_scaled = scaler.fit_transform(azdias_imputed)

print("Any NaN remaining?", np.isnan(azdias_scaled).any())

In [ ]:
pca_full = PCA(random_state=42)
azdias_pca_full = pca_full.fit_transform(azdias_scaled)

explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

plt.figure(figsize=(12, 5))
plt.plot(np.arange(1, len(cumulative) + 1), cumulative, marker='o', markersize=2)
plt.axhline(0.80, color='orange', linestyle='--', label='80% variance')
plt.axhline(0.90, color='red', linestyle='--', label='90% variance')
plt.xlabel('Number of principal components')
plt.ylabel('Cumulative explained variance')
plt.title('PCA Explained Variance')
plt.legend()
plt.show()

n_components_80 = int(np.argmax(cumulative >= 0.80) + 1)
n_components_90 = int(np.argmax(cumulative >= 0.90) + 1)
print("Components for 80% variance:", n_components_80)
print("Components for 90% variance:", n_components_90)

In [ ]:
# Choose a final component count after reviewing the variance plot.
N_COMPONENTS = n_components_80
pca = PCA(n_components=N_COMPONENTS, random_state=42)
azdias_pca = pca.fit_transform(azdias_scaled)
print(azdias_pca.shape)

In [ ]:
def component_weights(pca_model, component_index, columns, n=10):
    weights = pd.Series(pca_model.components_[component_index], index=columns)
    top_positive = weights.sort_values(ascending=False).head(n)
    top_negative = weights.sort_values(ascending=True).head(n)
    return pd.concat([top_positive, top_negative])

for i in range(3):
    print(f"
Principal Component {i+1}")
    display(component_weights(pca, i, azdias_clean.columns, n=5))

**Discussion:** Interpret at least three principal components. For each, describe the strongest positive and negative feature weights and the demographic meaning if clear.

## Step 3: Clustering

Fit KMeans models across a range of cluster counts and use the elbow in average distance/inertia to justify the final number of clusters.

In [ ]:
k_values = list(range(2, 21))
inertias = []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(azdias_pca)
    inertias.append(model.inertia_ / azdias_pca.shape[0])
    print(k, inertias[-1])

plt.figure(figsize=(10, 5))
plt.plot(k_values, inertias, marker='o')
plt.xlabel('Number of clusters')
plt.ylabel('Average point-centroid squared distance')
plt.title('KMeans Elbow Plot')
plt.show()

In [ ]:
# Choose final k after inspecting elbow plot.
N_CLUSTERS = 10
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
azdias_clusters = kmeans.fit_predict(azdias_pca)

population_cluster_props = pd.Series(azdias_clusters).value_counts(normalize=True).sort_index()
population_cluster_props

In [ ]:
# Apply the exact fitted imputer, scaler, PCA, and KMeans models to customers.
customers_imputed = imputer.transform(customers_clean)
customers_scaled = scaler.transform(customers_imputed)
customers_pca = pca.transform(customers_scaled)
customers_clusters = kmeans.predict(customers_pca)

customer_cluster_props = pd.Series(customers_clusters).value_counts(normalize=True).reindex(range(N_CLUSTERS), fill_value=0)
cluster_comparison = pd.DataFrame({
    'population_proportion': population_cluster_props.reindex(range(N_CLUSTERS), fill_value=0),
    'customer_proportion': customer_cluster_props,
})
cluster_comparison['difference'] = cluster_comparison['customer_proportion'] - cluster_comparison['population_proportion']
cluster_comparison['ratio'] = cluster_comparison['customer_proportion'] / cluster_comparison['population_proportion'].replace(0, np.nan)
cluster_comparison.sort_values('difference', ascending=False)

In [ ]:
cluster_comparison[['population_proportion', 'customer_proportion']].plot(kind='bar', figsize=(12, 5))
plt.ylabel('Proportion')
plt.title('Cluster Distribution: General Population vs Customers')
plt.show()

print("Overrepresented customer segments:")
display(cluster_comparison.sort_values('difference', ascending=False).head(3))
print("Underrepresented customer segments:")
display(cluster_comparison.sort_values('difference', ascending=True).head(3))

In [ ]:
def describe_cluster(cluster_id, n=10):
    """Describe a cluster center by inverse-transforming PCA/scaling into original feature space."""
    center_pca = kmeans.cluster_centers_[cluster_id].reshape(1, -1)
    center_scaled = pca.inverse_transform(center_pca)
    center_original = scaler.inverse_transform(center_scaled)[0]
    overall_mean = imputer.statistics_
    diff = pd.Series(center_original - overall_mean, index=azdias_clean.columns)
    return pd.concat([
        diff.sort_values(ascending=False).head(n),
        diff.sort_values(ascending=True).head(n)
    ])

for cluster_id in cluster_comparison.sort_values('difference', ascending=False).head(2).index:
    print(f"
Overrepresented cluster {cluster_id}")
    display(describe_cluster(cluster_id, n=5))

for cluster_id in cluster_comparison.sort_values('difference', ascending=True).head(2).index:
    print(f"
Underrepresented cluster {cluster_id}")
    display(describe_cluster(cluster_id, n=5))

## Final Discussion

Summarize:

- Which clusters are overrepresented among customers.
- Which clusters are underrepresented among customers.
- What demographic features characterize those segments.
- How these findings could guide mail-order campaign targeting.

Be careful not to overclaim; these are unsupervised clusters and should be treated as exploratory segmentation.

## Export

After all cells run successfully:

```bash
jupyter nbconvert --to html Identify_Customer_Segments.ipynb
zip arvato_customer_segments_submission.zip Identify_Customer_Segments.ipynb Identify_Customer_Segments.html
```